# Práctica 4: Modelo de Lenguaje Probabilista

Procesamiento de Lenguaje Natural


Eric Lemus Avalos

In [1]:
import os
import re
import nltk
import numpy as np
import random
import math 
# cargar los datos
def get_text_from_file(path_file: str, target_file: str) -> list:
    file_txt = []
    target_txt = []

    with open(path_file, 'r', encoding="utf8") as f_corpus, open(target_file, 'r', encoding="utf8") as f_target:
        for tweet in f_corpus:
            file_txt += [tweet]
        for target in f_target:
            target_txt += [target]
    
    file_txt = list(map(str.strip, file_txt))
    target_txt = list(map(int, target_txt))
    
    return file_txt, target_txt

In [2]:
txt_train, txt_y = get_text_from_file("../tarea_2/data_mex/mex20_train.txt", "../tarea_2/data_mex/mex20_train_labels.txt")
# file_txt_val, target_txt_val = get_text_from_file("../tarea_2/data_mex/mex20_val (1).txt", "../tarea_2/data_mex/mex20_val_labels (1).txt")


In [5]:
import nltk 
from nltk.tokenize import TweetTokenizer
tokenizer = TweetTokenizer()

corpus = []

for doc in txt_train:
    corpus += ["<s>"] + ["<s>"] + tokenizer.tokenize(doc) + ["</s>"] 
    


In [6]:
corpus

['<s>',
 '<s>',
 '@USUARIO',
 '@USUARIO',
 '@USUARIO',
 'Q',
 'se',
 'puede',
 'esperar',
 'del',
 'maricon',
 'de',
 'closet',
 'de',
 'la',
 'Yañez',
 'aun',
 'recuerdo',
 'esa',
 'ves',
 'q',
 'lo',
 'vi',
 'en',
 'zona',
 'rosa',
 'viendo',
 'quien',
 'lo',
 'levantada',
 '</s>',
 '<s>',
 '<s>',
 '@USUARIO',
 'La',
 'piel',
 'nueva',
 'siempre',
 'arde',
 'un',
 'poquito',
 'los',
 'primeros',
 'días',
 '...',
 'y',
 'más',
 'con',
 'este',
 'puto',
 'clima',
 '</s>',
 '<s>',
 '<s>',
 'Ustedes',
 'no',
 'se',
 'enamoran',
 'de',
 'mí',
 '…',
 'por',
 'tontas',
 '.',
 '</s>',
 '<s>',
 '<s>',
 'Me',
 'las',
 'va',
 'a',
 'pagar',
 'esa',
 'puta',
 'gorda',
 'roba',
 'tuits',
 '...',
 '</s>',
 '<s>',
 '<s>',
 '@USUARIO',
 'LA',
 'GENTE',
 'ES',
 'TONTA',
 'PORQUE',
 'NO',
 'SE',
 'DAN',
 'CUENTA',
 'QUE',
 'TÚ',
 'HACES',
 'A',
 'BATMAN',
 'AZUL',
 '</s>',
 '<s>',
 '<s>',
 'Estoy',
 'muy',
 'encabronada',
 'con',
 'las',
 'pseudo',
 'feministas',
 'por',
 'tontas',
 'e',
 'iletradas',

## Procesamiento 

In [260]:
from nltk.tokenize import TweetTokenizer
from nltk.probability import FreqDist

class TrigramData:
    
    def __init__(self, vocab_max: int, tk: TweetTokenizer):  
        self.vocab_max = vocab_max
        self.tk = tk
        self.UNK = "<unk>"
        self.EOS = "</s>"
        self.SOS = "<s>"
        self.final_vocab = set()
    
    def fit(self, raw_tokens: list) -> list:
        freq_dist = FreqDist()
        tokenized_corpus = []
        
        # Tokenizar tweets
        for txt in raw_tokens: 
            tokens = self.tk.tokenize(txt)
            tokenized_corpus.append(tokens)  # Guardar lista de tokens
            
            # Contar cada palabra en el vocabulario
            for word in tokens:  
                freq_dist[word] += 1  
        
        # Truncar el vocab a vocab_max
        self.final_vocab = {tok for tok,_ in freq_dist.most_common(self.vocab_max)}
        
        # Agregar tokens especiales
        self.final_vocab.update([self.UNK, self.SOS, self.EOS])
        
        transformed_corpus = [] 
        for tokens in tokenized_corpus:
            transformed_corpus.append(self.transform(tokens))
        
        return transformed_corpus 
        
    def mask_oov(self, word: str) -> str:
        return self.UNK if word not in self.final_vocab else word
        
    def add_sos_eos(self, tokens: list) -> list:
        return [self.SOS, self.SOS] + tokens + [self.EOS]
        
    def transform(self, tokens: list) -> list:
        transformed = []
        for word in tokens:
            transformed.append(self.mask_oov(word))  
        return self.add_sos_eos(transformed)  


## Trigram LM 

In [261]:
from typing import List, Tuple
class TrigramLM:

    def __init__(self, lambda1 = 0.4, lambda2 = 0.6, lambda3 = 0.1):
        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.lambda3 = lambda3

        #contadores
        self.unigram_counts = {}
        self.bigram_counts  = {}
        self.trigram_counts = {}

        self.vocab = 0
        self.total_tokes = 0
        self.V = 0

    def train(self, transformed_corpus, final_vocabulary):
        self.vocab =  final_vocabulary
        self.V = len(final_vocabulary)

        for tokens in transformed_corpus:
            for i, w in enumerate(tokens):

                #Unigrams 
                self.unigram_counts[w] = self.unigram_counts.get(w,0) + 1
                
                #Bigrams 
                if i >0: 
                    w_prev = tokens[i-1]
                    self.bigram_counts[(w_prev,w)] = self.bigram_counts.get((w_prev,w),0) + 1 
                
                #Trigams 
                if i > 1: 
                    w_prev_prev = tokens[i-2]
                    self.trigram_counts[(w_prev_prev, w_prev, w)] = \
                        self.trigram_counts.get((w_prev_prev, w_prev, w),0) + 1 

        self.total_tokes = sum(self.unigram_counts.values())


    def mask_oov(self, word):
        return "<unk>" if word not in self.vocab else word

    def unigram_probability(self, w):
        return (self.unigram_counts.get(self.mask_oov(w), 0) + 1) / (self.total_tokes + self.V) 

    def bigram_probability(self, w_prev, w):
        numerador = self.bigram_counts.get((self.mask_oov(w_prev), self.mask_oov(w)), 0) + 1
        denominador = self.total_tokes + self.V
        return  numerador / denominador
        
    def trigram_probability(self, w_prev_prev, w_prev, w):
        numerador = self.trigram_counts.get((self.mask_oov(w_prev_prev), self.mask_oov(w_prev), self.mask_oov(w)), 0) + 1
        denominador = self.total_tokes + self.V
        return  numerador / denominador

    def probability_word(self, w_prev_prev, w_prev, w):
        term_1 = self.lambda1 * self.unigram_probability(w)
        term_2 = self.lambda2 * self.bigram_probability(w_prev, w)
        term_3 = self.lambda3 * self.trigram_probability(w_prev_prev, w_prev, w)
        return  term_1 + term_2 + term_3

    def seq_probability(self,sequence):
        log_p = 0.0
        for i in range(len(sequence)):
            w_prev_prev = sequence[i-2] if i >= 2 else "<s>"
            w_prev = sequence[i-1] if i >= 1 else "<s>"
            w = sequence[i]
            p = self.probability_word(w_prev_prev, w_prev, w)
            log_p += math.log(p)
        return math.exp(log_p)
   
    def check_probas(self):
        return sum(self.unigram_probability(w) for w in self.vocab)

    def top_k_next_words(self, words: List[str], top: int) -> List[Tuple[str, float]]:
        candidates = {}
        w_prev_prev, w_prev = words[-2], words[-1]
        
        for word in self.vocab:
            prob = self.probability_word(w_prev_prev, w_prev, word)
            candidates[word] = prob
        
        return sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:top]

    def calc_perplexity(self, sequence: List[str]) -> float:
        N = len(sequence)
        log_prob_sum = sum(math.log(self.probability_word(sequence[i - 2], sequence[i - 1], sequence[i])) for i in range(2, N))
        return math.exp(-log_prob_sum / N)

    def generate_text(self, words: List[str], max_length: int) -> str:
        generated = words[:]
        for _ in range(max_length - len(words)):
            top_next = self.top_k_next_words(generated, top=1000)
            if not top_next:
                break
            next_word = random.choices([w for w, _ in top_next], weights=[p for _, p in top_next])[0]
            generated.append(next_word)
            if next_word == "</s>":
                break
        return " ".join(generated)


    def rank_permutation(self, test_tokens: list, top_k: int, botton_k: int):
        from itertools import permutations
        permuted_sequences = list(permutations(test_tokens))
        
        scored_sequences = [(seq, self.seq_probability(seq)) for seq in permuted_sequences]
        scored_sequences.sort(key=lambda x: x[1], reverse=True)
        
        top_sequences = scored_sequences[:top_k]
        bottom_sequences = scored_sequences[-botton_k:]
        
        return top_sequences, bottom_sequences

        

In [276]:
tk = TweetTokenizer()
trigram_data = TrigramData(vocab_max=15000, tk=tk)
transformed_corpus = trigram_data.fit(txt_train)
final_vacab =  trigram_data.final_vocab

In [105]:
tk = TweetTokenizer()
corpus = []
for file in txt_train:
    corpus += tk.tokenize(file)

fdist = nltk.FreqDist(corpus)

In [106]:
len(set(corpus))

15194

In [296]:
trigram_lm = TrigramLM(lambda1=0, lambda2=6, lambda3=1.0)

In [297]:
trigram_lm.train(transformed_corpus, final_vacab)

In [279]:
trigram_lm.check_probas()

1.0

## Pruebas 

In [293]:
w_prev_prev, w_prev, w = "<s>", "hola", "mundo"
p_w = trigram_lm.probability_word(w_prev_prev, w_prev, w)
print(f"\nP('{w}' | '{w_prev_prev}', '{w_prev}') = {p_w:.16f}")


P('mundo' | '<s>', 'hola') = 0.0004832047385239


In [281]:
w_prev_prev, w_prev, w = "<s>", "hijo", "de"
p_w = trigram_lm.probability_word(w_prev_prev, w_prev, w)
print(f"\nP('{w}' | '{w_prev_prev}', '{w_prev}') = {p_w:.16f}")


P('de' | '<s>', 'hijo') = 0.0046730574390149


In [282]:
w_prev_prev, w_prev, w = "hijo", "de", "la"
p_w = trigram_lm.probability_word(w_prev_prev, w_prev, w)
print(f"\nP('{w}' | '{w_prev_prev}', '{w_prev}') = {p_w:.16f}")


P('la' | 'hijo', 'de') = 0.0141625750136388


In [283]:
w_prev_prev, w_prev, w = "vete", "a", "la"
p_w = trigram_lm.probability_word(w_prev_prev, w_prev, w)
print(f"\nP('{w}' | '{w_prev_prev}', '{w_prev}') = {p_w:.16f}")


P('la' | 'vete', 'a') = 0.0148951757462396


### Top next words

In [285]:
def top_5(tokens_test, trigram_lm):
    top_5 = trigram_lm.top_k_next_words(words=tokens_test, top=5)
    print(f"\nTop 5 sucesores de ({tokens_test[0]}, {tokens_test[1]}): ")
    for cand, sc in top_5: 
        print(f"  {cand} -> {sc:.16f}")

In [298]:
tokens_test = ["hijo", "de"]
top_5(tokens_test,trigram_lm)


Top 5 sucesores de (hijo, de): 
  la -> 0.0125555295768062
  mi -> 0.0037487335359676
  los -> 0.0034681630426311
  su -> 0.0029927519289221
  que -> 0.0029537838048476


In [299]:
tokens_test = ["hijo", "su"]
top_5(tokens_test,trigram_lm)


Top 5 sucesores de (hijo, su): 
  madre -> 0.0038890187826358
  puta -> 0.0014106460914972
  hijo -> 0.0003818876159302
  verga -> 0.0003351258670408
  novia -> 0.0003351258670408


In [300]:
tokens_test = ["america", "es"]
top_5(tokens_test,trigram_lm)


Top 5 sucesores de (america, es): 
  que -> 0.0035616865404099
  la -> 0.0022991193203959
  un -> 0.0021120723248383
  una -> 0.0015041695892760
  el -> 0.0015041695892760


In [301]:
tokens_test = ["vete", "a"]
top_5(tokens_test,trigram_lm)


Top 5 sucesores de (vete, a): 
  la -> 0.0132725430597771
  su -> 0.0038890187826358
  las -> 0.0037019717870782
  los -> 0.0035149247915205
  mi -> 0.0033278777959629


### Probabilidad de sequencia

In [302]:
seq_example = ["hola", "como", "has", "estado", "</s>"]
seq_prob = trigram_lm.seq_probability(seq_example)
print(f"\n Prob. de la secuencia '{' '.join(seq_example)}': {seq_prob:.20f}")



 Prob. de la secuencia 'hola como has estado </s>': 0.00000000000000000000


In [303]:
seq_example = ["arriba", "el", "america", "</s>"]
seq_prob = trigram_lm.seq_probability(seq_example)
print(f"\n Prob. de la secuencia '{' '.join(seq_example)}': {seq_prob:.20f}")


 Prob. de la secuencia 'arriba el america </s>': 0.00000000000000009491


In [304]:
seq_example = ["vete", "a", "la", "verga"]
seq_prob = trigram_lm.seq_probability(seq_example)
print(f"\n Prob. de la secuencia '{' '.join(seq_example)}': {seq_prob:.20f}")


 Prob. de la secuencia 'vete a la verga': 0.00000000000305997333


### Generar Texto

In [305]:
tokens = ["vete", "a"]
text_generated = trigram_lm.generate_text(tokens, 10)
print(f"\nTexto generado desde los tokens '{' '.join(tokens)}': {text_generated}")


Texto generado desde los tokens 'vete a': vete a saber Pame lava Huevo mg amY Carmona promete


In [306]:
tokens = ["vete", "a"]
text_generated = trigram_lm.generate_text(tokens, 10)
print(f"\nTexto generado desde los tokens '{' '.join(tokens)}': {text_generated}")


Texto generado desde los tokens 'vete a': vete a la cuelgue GAY dilo adivinen #Keep_Mwave_EXO altar avisar


### Perplexity

In [307]:
seq_example = ["hola", "como", "has", "estado", "</s>"]
perplexity = trigram_lm.calc_perplexity(seq_example)
print(f"\n Perplexity de la secuencia '{' '.join(seq_example)}': {perplexity:.20f}")


 Perplexity de la secuencia 'hola como has estado </s>': 361.32455707556459856278


In [308]:
seq_example = ["hola", "vete", "a", "la", "verga"]
perplexity = trigram_lm.calc_perplexity(seq_example)
print(f"\n Perplexity de la secuencia '{' '.join(seq_example)}': {perplexity:.20f}")


 Perplexity de la secuencia 'hola vete a la verga': 28.19836831609035954216


### Permutaciones 

In [311]:
test_tokens = ["sino", "gano", "me", "voy", "a", "la", "chingada"]
top, bottom = trigram_lm.rank_permutation(test_tokens,5,5)

print("\nMejores 5 permutaciones")
for seq in top:
    print(f"'{' '.join(seq[0])}': {seq[1]}")

print("\nPeores 5 permutaciones")
for seq in bottom:
    print(f"'{' '.join(seq[0])}': {seq[1]}")


Mejores 5 permutaciones
'me voy a la chingada sino gano': 2.6510321634633034e-22
'me voy a la chingada gano sino': 2.6510321634633034e-22
'sino gano me voy a la chingada': 2.024689399568107e-23
'sino me voy a la chingada gano': 2.024689399568107e-23
'gano sino me voy a la chingada': 2.024689399568107e-23

Peores 5 permutaciones
'voy chingada me gano la a sino': 2.671210157909182e-30
'voy chingada me a sino gano la': 2.671210157909182e-30
'voy chingada me a sino la gano': 2.671210157909182e-30
'voy chingada me a gano sino la': 2.671210157909182e-30
'voy chingada me a gano la sino': 2.671210157909182e-30
